# ASOS → OpenSense Conversion

Fetch ASOS 1-minute weather data from the IEM API and convert to
OpenSense-format NetCDF.

**Source:** https://mesonet.agron.iastate.edu/request/asos/1min.phtml  
**Output format:** OpenSense v1.0 — dims `(id, time)`, compressed NetCDF

| Step | Cell |
|------|------|
| 1 | Imports & config |
| 2 | Discover NYC stations |
| 3 | Fetch raw data |
| 4 | Inspect single station |
| 5 | Convert to OpenSense dataset |
| 6 | Save NetCDF |

## 1. Imports & config

In [1]:
import warnings
from datetime import datetime

import pandas as pd
import xarray as xr

from OpenMesh.notebooks.ws_opensense_conversion import (
    # paths
    SAMPLE_DIR,  # station discovery
    fetch_asos_stations_nyc,
    load_asos_metadata,
    # fetch
    fetch_asos_raw,
    # convert + save
    to_opensense_dataset,
    save_opensense_dataset,
)

warnings.filterwarnings('ignore')

# ── time window ───────────────────────────────────────────────
START_DT = datetime(2024, 1, 15)
END_DT   = datetime(2024, 1, 22)

# ── variables to fetch ────────────────────────────────────────
# full list: print(list(ASOS_AVAILABLE_VARS.keys()))
VARIABLES = [
    'precip_amount',
    'precip_type',
    'precip_category',
    'temperature',
    'wind_speed',
    'wind_direction',
]

print(f'Period    : {START_DT.date()}  →  {END_DT.date()}')
print(f'Variables : {VARIABLES}')
print(f'SAMPLE_DIR: {SAMPLE_DIR.resolve()}')

Period    : 2024-01-15  →  2024-01-22
Variables : ['precip_amount', 'precip_type', 'precip_category', 'temperature', 'wind_speed', 'wind_direction']
SAMPLE_DIR: /Users/drorjac/PycharmProjects/opensense_example_data_openmesh/OpenMesh/notebooks/data/samples


## 2. Discover NYC-area ASOS stations

In [2]:
# queries IEM network API + saves to ASOS_META_PATH
# re-run to refresh; loads from CSV if already saved:
# meta = load_asos_metadata()

meta     = fetch_asos_stations_nyc()
stations = meta.index.tolist()

print(f'\nAvailable ({len(stations)}): {stations}')

  Querying NY_ASOS ... 7 stations found
  Querying NJ_ASOS ... 5 stations found
  Querying CT_ASOS ... 1 stations found

  Total : 13 stations
  ID       Network    Name                                    Lat      Lon   Elev
  ────────────────────────────────────────────────────────────────────────
  BDR      CT_ASOS    Bridgeport/Sikorsky                  41.158  -73.129    5.0
  CDW      NJ_ASOS    CALDWELL/ESSEX CO.                   40.876  -74.283   53.0
  TEB      NJ_ASOS    TETERBORO AIRPORT                    40.859  -74.056    3.0
  MMU      NJ_ASOS    MORRISTOWN MUNI                      40.799  -74.415   57.0
  EWR      NJ_ASOS    Newark Intl                          40.683  -74.169    2.0
  LDJ      NJ_ASOS    Linden                               40.617  -74.245    7.0
  HPN      NY_ASOS    WHITE PLAINS                         41.067  -73.707  134.0
  ISP      NY_ASOS    ISLIP/MACARTHUR                      40.794  -73.102   30.0
  LGA      NY_ASOS    New York/LaGuardia    

In [3]:


asos_meta = load_asos_metadata()
# IDs must match the index: CDW, TEB, MMU, EWR, LDJ, LGA, NYC, JRB, JFK
selected_stations = ['JFK', 'EWR', 'LGA', 'NYC']

print(f"Selected: {selected_stations}")
print(asos_meta.loc[selected_stations])

Selected: ['JFK', 'EWR', 'LGA', 'NYC']
                           Name  Latitude  Longitude  Elevation  Network
Station ID                                                              
JFK         NEW YORK/JF KENNEDY   40.6386   -73.7622        7.0  NY_ASOS
EWR                 Newark Intl   40.6827   -74.1693        2.0  NJ_ASOS
LGA          New York/LaGuardia   40.7794   -73.8803        9.0  NY_ASOS
NYC               NEW YORK CITY   40.7790   -73.9692       27.0  NY_ASOS


## 3. Fetch raw ASOS data

In [4]:
# If you want a shorter window, define END_DT as datetime, not string:
END_DT = datetime(2024, 1, 17)
asos_raw = fetch_asos_raw(
    stations=selected_stations,
    start_dt=START_DT,
    end_dt=END_DT,
    variables=VARIABLES,
)

print(f'\nFetched: {list(asos_raw.keys())}')

  Fetching JFK ... 2,760 records
  Fetching EWR ... 1,392 records
  Fetching LGA ... 2,880 records
  Fetching NYC ... 1,795 records

Fetched: ['JFK', 'EWR', 'LGA', 'NYC']


## 4. Inspect single station

In [5]:
station_id = 'JFK'   # change to any fetched station

if station_id not in asos_raw:
    print(f'Not in asos_raw. Available: {list(asos_raw.keys())}')
else:
    df = asos_raw[station_id]

    # metadata
    if station_id in meta.index:
        row = meta.loc[station_id]
        print(f'Station  : {station_id}  —  {row["Name"]}')
        print(f'Lat/Lon  : {row["Latitude"]:.4f}, {row["Longitude"]:.4f}')
        print(f'Elev     : {row["Elevation"]:.1f} m')
        print(f'Network  : {row["Network"]}')

    # data
    print(f'\nRecords  : {len(df):,}')
    print(f'Time     : {str(df.index[0])[:19]}  →  {str(df.index[-1])[:19]}')
    print(f'Columns  : {df.columns.tolist()}')
    if 'precip_amount' in df.columns:
        print(f'Rain sum : {df["precip_amount"].sum():.2f} mm')
    print()
    print(df.head(10))

Station  : JFK  —  NEW YORK/JF KENNEDY
Lat/Lon  : 40.6386, -73.7622
Elev     : 7.0 m
Network  : NY_ASOS

Records  : 2,760
Time     : 2024-01-15 00:00:00  →  2024-01-16 23:59:00
Columns  : ['station_id', 'precip_amount', 'precip_type', 'precip_category', 'temperature', 'wind_speed', 'wind_direction']
Rain sum : 10.16 mm

                    station_id  precip_amount precip_type precip_category  \
datetime                                                                    
2024-01-15 00:00:00        JFK            0.0          NP             dry   
2024-01-15 00:01:00        JFK            0.0          NP             dry   
2024-01-15 00:02:00        JFK            0.0          NP             dry   
2024-01-15 00:03:00        JFK            0.0          NP             dry   
2024-01-15 00:04:00        JFK            0.0          NP             dry   
2024-01-15 00:05:00        JFK            0.0          NP             dry   
2024-01-15 00:06:00        JFK            0.0          NP     

## 5. Convert to OpenSense dataset

In [6]:
ds_asos = to_opensense_dataset(
    data=asos_raw,
    source='asos',
    variables=VARIABLES,
    meta=meta,
    start_dt=START_DT,
    end_dt=END_DT,
)

print(f'Shape    : {dict(ds_asos.sizes)}')
print(f'Time     : {str(ds_asos.time.values[0])[:19]}  →  {str(ds_asos.time.values[-1])[:19]}')
print(f'Stations : {ds_asos.id.values}')
print(f'Vars     : {list(ds_asos.data_vars)}')
print(ds_asos)

Shape    : {'id': 4, 'time': 2880}
Time     : 2024-01-15T00:00:00  →  2024-01-16T23:59:00
Stations : ['JFK' 'EWR' 'LGA' 'NYC']
Vars     : ['precip_amount', 'precip_type', 'precip_category', 'temperature', 'wind_speed', 'wind_direction']
<xarray.Dataset> Size: 576kB
Dimensions:          (id: 4, time: 2880)
Coordinates:
  * id               (id) <U3 48B 'JFK' 'EWR' 'LGA' 'NYC'
  * time             (time) datetime64[ns] 23kB 2024-01-15 ... 2024-01-16T23:...
    lat              (id) float64 32B 40.64 40.68 40.78 40.78
    lon              (id) float64 32B -73.76 -74.17 -73.88 -73.97
    elev             (id) float64 32B 7.0 2.0 9.0 27.0
Data variables:
    precip_amount    (id, time) float64 92kB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    precip_type      (id, time) object 92kB 'NP' 'NP' 'NP' ... 'NP' 'NP' 'NP'
    precip_category  (id, time) object 92kB 'dry' 'dry' 'dry' ... 'dry' 'dry'
    temperature      (id, time) float64 92kB 0.0 0.0 0.0 ... -2.222 -2.222
    wind_speed       (id, time

## 6. Save NetCDF (multi-station + one file per station)

In [7]:
# Set Global Attributes

# ── note on naming ────────────────────────────────────────────
# ASOS data keeps 'precip_amount' / 'precip_rate' intentionally —
# OpenSense PWS convention uses 'rainfall_amount' / 'rainfall_rate'
# but ASOS measures all precipitation types (rain, snow, etc.)
# so 'precip' is more accurate here. No renaming applied.

GLOBAL_ATTRS = {
    "title":        "OpenMesh ASOS reference precipitation — NYC",
    "institution":  "NOAA / IEM",
    "source":       "NOAA ASOS 1-min via Iowa Environmental Mesonet",
    "references":   "https://mesonet.agron.iastate.edu/request/asos/1min.phtml",
    "history":      f"Created {pd.Timestamp.now().strftime('%Y-%m-%d')}",
    "comment":      "ASOS airport stations — precip_amount/rate kept (not renamed to rainfall_* since ASOS captures all precip types)",
    "license":      "Public domain (NOAA)",
    "conventions":  "OpenSense-PWS-v1.0",
    "featureType":  "timeSeries",
    "sensor_type":  "ASOS",
    "time_zone":    "UTC",
}


In [8]:
# ── date range for filenames ───────────────────────────────────
_date_start = START_DT.strftime("%Y-%m-%d")
_date_end   = END_DT.strftime("%Y-%m-%d")

# ── save multi-station (one flat file) + one file per station ──
ds_asos_sample = to_opensense_dataset(
    data=asos_raw, source="asos",
    variables=VARIABLES, meta=meta,
    start_dt=START_DT, end_dt=END_DT,
    extra_attrs=GLOBAL_ATTRS,
)
path_asos = save_opensense_dataset(
    ds_asos_sample, output_dir=SAMPLE_DIR,
    filename=f"ASOS_{_date_start}_{_date_end}.nc",
)

print(f"  Multi-station : {path_asos}")
print(f"  Shape        : {dict(ds_asos_sample.sizes)}")
print(f"  Stations     : {list(ds_asos_sample.id.values)}")
print(f"  Vars         : {list(ds_asos_sample.data_vars)}")
print(ds_asos_sample.sel(id="JFK"))

# ── one file per station (station name in filename) ───────────
paths_single = []
for sid in ds_asos_sample.id.values:
    ds_one = to_opensense_dataset(
        data={sid: asos_raw[str(sid)]}, source="asos",
        variables=VARIABLES, meta=meta,
        start_dt=START_DT, end_dt=END_DT,
        extra_attrs=GLOBAL_ATTRS,
    )
    p = save_opensense_dataset(
        ds_one, output_dir=SAMPLE_DIR,
        filename=f"ASOS_{sid}_{_date_start}_{_date_end}.nc",
    )
    paths_single.append(p)
print(f"  Single-station files : {[p.name for p in paths_single]}")


  Saved : ASOS_2024-01-15_2024-01-17.nc  (1.34 MB)
  Multi-station : /Users/drorjac/PycharmProjects/opensense_example_data_openmesh/OpenMesh/notebooks/data/samples/ASOS_2024-01-15_2024-01-17.nc
  Shape        : {'id': 4, 'time': 2880}
  Stations     : [np.str_('JFK'), np.str_('EWR'), np.str_('LGA'), np.str_('NYC')]
  Vars         : ['precip_amount', 'precip_type', 'precip_category', 'temperature', 'wind_speed', 'wind_direction']
<xarray.Dataset> Size: 161kB
Dimensions:          (time: 2880)
Coordinates:
    id               <U3 12B 'JFK'
  * time             (time) datetime64[ns] 23kB 2024-01-15 ... 2024-01-16T23:...
    lat              float64 8B 40.64
    lon              float64 8B -73.76
    elev             float64 8B 7.0
Data variables:
    precip_amount    (time) float64 23kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    precip_type      (time) object 23kB 'NP' 'NP' 'NP' 'NP' ... 'NP' 'NP' 'NP'
    precip_category  (time) object 23kB 'dry' 'dry' 'dry' ... 'dry' 'dry' 'dry'
    te

## 7. Reload & verify

In [10]:
ds_check = xr.open_dataset(path_asos)

print(f'Shape    : {dict(ds_check.sizes)}')
print(f'Stations : {ds_check.id.values}')
print(f'Vars     : {list(ds_check.data_vars)}')
print(ds_check)

Shape    : {'id': 4, 'time': 2880}
Stations : ['JFK' 'EWR' 'LGA' 'NYC']
Vars     : ['precip_amount', 'precip_type', 'precip_category', 'temperature', 'wind_speed', 'wind_direction']
<xarray.Dataset> Size: 807kB
Dimensions:          (id: 4, time: 2880)
Coordinates:
  * id               (id) <U3 48B 'JFK' 'EWR' 'LGA' 'NYC'
  * time             (time) datetime64[ns] 23kB 2024-01-15 ... 2024-01-16T23:...
    lat              (id) float64 32B ...
    lon              (id) float64 32B ...
    elev             (id) float64 32B ...
Data variables:
    precip_amount    (id, time) float64 92kB ...
    precip_type      (id, time) <U2 92kB ...
    precip_category  (id, time) <U7 323kB ...
    temperature      (id, time) float64 92kB ...
    wind_speed       (id, time) float64 92kB ...
    wind_direction   (id, time) float64 92kB ...
Attributes: (12/14)
    title:        OpenMesh ASOS reference precipitation — NYC
    source:       NOAA ASOS 1-min via Iowa Environmental Mesonet
    start_date:   20